# 03.5.2 FastF1 Data Reproduction (Clean)

Barebones reproduction pipeline with standardized conversions (times, milliseconds, positions) and a simple match check vs master 2024.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
KAGGLE_DIR = RAW_DATA_DIR / 'kaggle'
FASTF1_DIR = RAW_DATA_DIR / 'fastf1_2018plus'

circuits_df = pd.read_csv(KAGGLE_DIR / 'circuits.csv', low_memory=False)
drivers_df = pd.read_csv(KAGGLE_DIR / 'drivers.csv', low_memory=False)
status_df = pd.read_csv(KAGGLE_DIR / 'status.csv', low_memory=False)
races_df = pd.read_csv(KAGGLE_DIR / 'races.csv', low_memory=False)
master_df = pd.read_csv(PROCESSED_DATA_DIR / 'master_races_clean.csv', low_memory=False)

status_lookup = dict(zip(status_df['status'].str.strip(), status_df['statusId']))
driver_code_to_id = dict(zip(drivers_df['code'].astype(str).str.strip().str.upper(), drivers_df['driverId']))
constructor_lookup = dict(
    ((int(row['year']), str(row['code']).strip().upper()), int(row['constructorId']))
    for _, row in master_df[['year', 'code', 'constructorId']].drop_duplicates(['year', 'code']).iterrows()
)



In [6]:
# Helper functions

def parse_time_to_td(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    if s in ["", "\\N", "NaT", "None", "nan"]:
        return pd.NaT
    s = s.replace("\u202f", "").replace("\xa0", "").strip()
    if "days" in s:
        s = s.split("days", 1)[1].strip()
    if re.match(r"^\d+:\d{2}\.\d+$", s) or re.match(r"^\d+:\d{2}\.\d{3,6}$", s):
        s = "00:" + s
    elif re.match(r"^\d+\.\d+$", s):
        s = "00:00:" + s
    elif re.match(r"^\d+:\d{2}:\d{2}$", s):
        s = s + ".000"
    try:
        return pd.to_timedelta(s).round("1ms")
    except Exception:
        return pd.NaT

def td_to_ms(td):
    if pd.isna(td):
        return pd.NA
    return int(round(td.total_seconds() * 1000))

def normalize_quali(td_val):
    td = parse_time_to_td(td_val)
    if pd.isna(td):
        return pd.NA
    total_seconds = td.total_seconds()
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f"{minutes}:{seconds:05.3f}"

def map_status(status_text):
    if pd.isna(status_text):
        return pd.NA
    return status_lookup.get(str(status_text).strip(), pd.NA)

def convert_gap_times_to_absolute_ms(df):
    if df.empty:
        return df
    df = df.copy()
    df['time_td'] = df['time'].apply(parse_time_to_td)
    leader_ms = {}
    # Use 'year' and 'name' instead of 'Year' and 'Event'
    for (y, ev), grp in df.groupby(['year', 'name']):
        leader_rows = grp[grp['position'] == 1]
        if leader_rows.empty:
            continue
        lt = leader_rows.iloc[0]['time_td']
        if pd.isna(lt):
            continue
        leader_ms[(y, ev)] = td_to_ms(lt)
    converted_ms = []
    for _, row in df.iterrows():
        td = row['time_td']
        if pd.isna(td):
            converted_ms.append(pd.NA)
            continue
        ms = td_to_ms(td)
        # Use 'year' and 'name' instead of 'Year' and 'Event'
        key = (row['year'], row['name'])
        lm = leader_ms.get(key)
        if row['position'] != 1 and lm is not None and ms < 60000:
            abs_ms = lm + ms
            converted_ms.append(abs_ms)
        else:
            converted_ms.append(ms)
    df['milliseconds'] = converted_ms
    def ms_to_time_str(ms):
        if pd.isna(ms):
            return pd.NA
        total_seconds = ms / 1000
        hours = int(total_seconds // 3600)
        minutes = int((total_seconds % 3600) // 60)
        seconds = total_seconds % 60
        if hours > 0:
            return f"{hours}:{minutes:02d}:{seconds:06.3f}"
        else:
            return f"{minutes}:{seconds:06.3f}"
    df['time'] = df['milliseconds'].apply(ms_to_time_str)
    df = df.drop(columns=['time_td'])
    return df



In [7]:
def load_fastf1_year(year):
    res = pd.read_csv(FASTF1_DIR / f"ALL_RESULTS_{year}.csv", low_memory=False)
    laps = pd.read_csv(FASTF1_DIR / f"ALL_LAPS_{year}.csv", low_memory=False)
    return {'results': res, 'laps': laps}


def extract_race_data(fastf1_data, year):
    res = fastf1_data['results']
    race = res[res['Session'] == 'R'].copy()
    race = race.rename(columns={
        'Year': 'year',
        'Event': 'name',
        'Abbreviation': 'code',
        'GridPosition': 'grid',
        'Position': 'position',
        'Points': 'points',
        'Laps': 'laps',
        'Time': 'time',
        'Status': 'status_text'
    })
    race['code'] = race['code'].astype(str).str.strip().str.upper()
    race['grid'] = pd.to_numeric(race['grid'], errors='coerce').astype('Int64')
    race['position'] = pd.to_numeric(race['position'], errors='coerce').astype('Int64')
    race['points'] = pd.to_numeric(race['points'], errors='coerce')
    race['laps'] = pd.to_numeric(race['laps'], errors='coerce').astype('Int64')
    race['statusId'] = race['status_text'].apply(map_status)
    race = convert_gap_times_to_absolute_ms(race)
    race['Year'] = race['year']
    race['Event'] = race['name']
    race['year'] = race['year'].astype(int)
    race['driverId'] = race['code'].map(driver_code_to_id)
    return race


def map_circuit_info(race_df):
    race_lookup = {(int(r['year']), str(r['name']).strip()): r for _, r in races_df.iterrows()}
    circuit_coords = dict(zip(circuits_df['circuitId'], zip(circuits_df['lat'], circuits_df['lng'])))
    def lookup(row):
        info = race_lookup.get((int(row['year']), str(row['name']).strip()), {})
        row['circuitId'] = info.get('circuitId', pd.NA)
        row['date'] = pd.to_datetime(info.get('date', pd.NaT))
        row['round'] = info.get('round', pd.NA)
        if pd.notna(row['circuitId']):
            latlng = circuit_coords.get(row['circuitId'], (pd.NA, pd.NA))
            row['lat'], row['lng'] = latlng
        else:
            row['lat'] = pd.NA
        return row
    return race_df.apply(lookup, axis=1)


def calculate_fastest_lap(race_df, laps_df):
    laps = laps_df[laps_df['Session'] == 'R'].copy()
    laps['LapTime_td'] = laps['LapTime'].apply(parse_time_to_td)
    valid = laps[laps['LapTime_td'].notna() & (laps['LapTime_td'] > pd.Timedelta(0))]
    if valid.empty:
        race_df['fastestLap'] = pd.NA
        race_df['fastestLapTime'] = pd.NA
        return race_df
    best_idx = valid.groupby(['Year', 'Event', 'Driver'])['LapTime_td'].idxmin()
    best = valid.loc[best_idx, ['Year', 'Event', 'Driver', 'LapNumber', 'LapTime_td']].copy()
    best = best.rename(columns={'Driver': 'code', 'LapNumber': 'fastestLap', 'LapTime_td': 'fastestLap_td'})
    best['code'] = best['code'].astype(str).str.strip().str.upper()
    best['fastestLapTime'] = best['fastestLap_td'].apply(lambda td: f"{int(td.total_seconds()//60)}:{td.total_seconds()%60:05.3f}" if pd.notna(td) else pd.NA)
    race_df = race_df.merge(best[['Year', 'Event', 'code', 'fastestLap', 'fastestLapTime']],
                             left_on=['year', 'name', 'code'], right_on=['Year', 'Event', 'code'], how='left')
    race_df = race_df.drop(columns=['Year_y', 'Event_y'], errors='ignore')
    return race_df


def calculate_fastest_lap_speed(race_df, laps_df):
    laps = laps_df[laps_df['Session'] == 'R'].copy()
    if 'SpeedFL' not in laps.columns:
        race_df['fastestLapSpeed'] = pd.NA
        return race_df
    speeds = laps[['Year', 'Event', 'Driver', 'LapNumber', 'SpeedFL']].copy()
    speeds['Driver'] = speeds['Driver'].astype(str).str.strip().str.upper()
    race_df = race_df.merge(
        speeds,
        left_on=['year', 'name', 'code', 'fastestLap'],
        right_on=['Year', 'Event', 'Driver', 'LapNumber'],
        how='left'
    )
    race_df = race_df.rename(columns={'SpeedFL': 'fastestLapSpeed'})
    race_df = race_df.drop(columns=['Year_y', 'Event_y', 'Driver', 'LapNumber'], errors='ignore')
    return race_df


def add_sprint_results(race_df, fastf1_data):
    res = fastf1_data['results']
    laps = fastf1_data['laps']
    sprint_res = res[res['Session'] == 'Sprint'].copy()
    if sprint_res.empty:
        for col in ['sprint_results_grid','sprint_results_positionOrder','sprint_results_points','sprint_results_laps','sprint_results_time','sprint_results_milliseconds','sprint_results_fastestLap','sprint_results_fastestLapTime','sprint_results_statusId']:
            race_df[col] = pd.NA
        return race_df
    sprint_res = sprint_res.rename(columns={
        'Year':'year','Event':'name','Abbreviation':'code',
        'GridPosition':'sprint_results_grid','Position':'sprint_results_positionOrder',
        'Points':'sprint_results_points','Status':'status_text'
    })
    sprint_res['code'] = sprint_res['code'].astype(str).str.strip().str.upper()
    sprint_res['sprint_results_statusId'] = sprint_res['status_text'].apply(map_status)
    sprint_res = sprint_res.drop(columns=['status_text'])
    sprint_laps = laps[laps['Session']=='Sprint'].copy()
    if not sprint_laps.empty:
        sprint_laps['LapTime_td'] = sprint_laps['LapTime'].apply(parse_time_to_td)
        sprint_laps['Driver'] = sprint_laps['Driver'].astype(str).str.strip().str.upper()
        lap_counts = sprint_laps.groupby(['Year','Event','Driver']).size().reset_index(name='sprint_results_laps')
        lap_sums = sprint_laps.groupby(['Year','Event','Driver'])['LapTime_td'].sum().reset_index()
        lap_sums['sprint_results_milliseconds'] = lap_sums['LapTime_td'].apply(td_to_ms)
        best_idx = sprint_laps[sprint_laps['LapTime_td'].notna()].groupby(['Year','Event','Driver'])['LapTime_td'].idxmin()
        best = sprint_laps.loc[best_idx, ['Year','Event','Driver','LapNumber','LapTime']].copy()
        best = best.rename(columns={'LapNumber':'sprint_results_fastestLap','LapTime':'sprint_results_fastestLapTime'})
        for dfm in [lap_counts, lap_sums, best]:
            dfm['Driver'] = dfm['Driver'].astype(str).str.strip().str.upper()
        sprint_res = sprint_res.merge(lap_counts.rename(columns={'Driver':'code','Year':'year','Event':'name'}), on=['year','name','code'], how='left')
        sprint_res = sprint_res.merge(lap_sums.rename(columns={'Driver':'code','Year':'year','Event':'name','LapTime_td':'sprint_results_time'}), on=['year','name','code'], how='left')
        sprint_res = sprint_res.merge(best.rename(columns={'Driver':'code','Year':'year','Event':'name'}), on=['year','name','code'], how='left')
    race_df = race_df.merge(sprint_res, on=['year','name','code'], how='left')
    return race_df


def calculate_standings(race_df):
    if race_df.empty:
        return race_df
    race_df = race_df.sort_values(['year','round','date']).reset_index(drop=True)
    race_df['points_total'] = race_df['points'].fillna(0)
    if 'sprint_results_points' in race_df.columns:
        race_df['points_total'] = race_df['points_total'] + race_df['sprint_results_points'].fillna(0)
    race_df['driver_standings_points'] = race_df.groupby('driverId')['points_total'].cumsum()
    race_df['driver_standings_position'] = race_df.groupby(['year','round'])['driver_standings_points'].rank(method='min', ascending=False).astype(int)
    cons_points = race_df.groupby(['year','round','constructorId'])['driver_standings_points'].sum().reset_index()
    cons_points = cons_points.rename(columns={'driver_standings_points':'constructor_standings_points'})
    race_df = race_df.merge(cons_points, on=['year','round','constructorId'], how='left')
    race_df['constructor_standings_position'] = race_df.groupby(['year','round'])['constructor_standings_points'].rank(method='min', ascending=False).astype(int)
    race_df['driver_standings_points_PRE_RACE'] = race_df.groupby('driverId')['driver_standings_points'].shift(1)
    race_df['driver_standings_position_PRE_RACE'] = race_df.groupby('driverId')['driver_standings_position'].shift(1)
    cons_pre = race_df.groupby(['year','round','constructorId'])['driver_standings_points_PRE_RACE'].sum().reset_index()
    cons_pre = cons_pre.rename(columns={'driver_standings_points_PRE_RACE':'constructor_standings_points_PRE_RACE'})
    race_df = race_df.merge(cons_pre, on=['year','round','constructorId'], how='left')
    race_df['constructor_standings_position_PRE_RACE'] = race_df.groupby(['year','round'])['constructor_standings_points_PRE_RACE'].rank(method='min', ascending=False)
    return race_df



In [12]:
def reproduce_year(year):
    fdata = load_fastf1_year(year)
    race = extract_race_data(fdata, year)
    race = map_circuit_info(race)
    race['constructorId'] = race.apply(lambda r: constructor_lookup.get((int(r['year']), r['code']), pd.NA), axis=1)
    race = calculate_fastest_lap(race, fdata['laps'])
    race = calculate_fastest_lap_speed(race, fdata['laps'])
    race = add_sprint_results(race, fdata)
    race = calculate_standings(race)
    race['podium'] = race['position'].apply(lambda x: 1 if pd.notna(x) and x in [1,2,3] else 0)
    race['rank'] = race['position']
    for col in ['q1','q2','q3']:
        if col in race.columns:
            race[col] = race[col].apply(normalize_quali)
        else:
            race[col] = pd.NA
    dob_lookup = dict(zip(drivers_df['code'].astype(str).str.strip().str.upper(), pd.to_datetime(drivers_df['dob'])))
    race['driver_age'] = (race['date'] - race['code'].map(dob_lookup)).dt.days / 365.25
    race['status_category'] = race['statusId'].apply(lambda sid: 'Unknown' if pd.isna(sid) else ('Finished' if sid==1 else ('Finished_Lapped' if sid in [11,12,13,14,15,16,17,18,19,20,45,50,53,55,58,88,111,112,113,114,115,116,117,118,119,120,122,123,124,125,127,133,134] else ('Disqualified' if sid==2 else ('Not_Classified' if sid==62 else 'DNF')))))
    
    # Drop any duplicate Year/Event columns that might have been created from merges
    race = race.drop(columns=['Year', 'Event'], errors='ignore')
    # Also drop any _x, _y suffixed columns from merges
    race = race.loc[:, ~race.columns.str.endswith('_x')]
    race = race.loc[:, ~race.columns.str.endswith('_y')]
    
    reproduced_cols = ['resultId','raceId','driverId','constructorId','grid','position','points','laps','time','milliseconds','fastestLap','rank','fastestLapTime','fastestLapSpeed','statusId','year','round','circuitId','date','name','lat','lng','code','driver_standings_points','driver_standings_position','constructor_standings_points','constructor_standings_position','q1','q2','q3','sprint_results_grid','sprint_results_positionOrder','sprint_results_points','sprint_results_laps','sprint_results_time','sprint_results_milliseconds','sprint_results_fastestLap','sprint_results_fastestLapTime','sprint_results_statusId','podium','driver_standings_points_PRE_RACE','driver_standings_position_PRE_RACE','constructor_standings_position_PRE_RACE','driver_age','status_category']
    for col in reproduced_cols:
        if col not in race.columns:
            race[col] = pd.NA
    race = race[reproduced_cols]
    return race

In [ ]:
# Reproduce 2024 and compute match %
repro_2024 = reproduce_year(2024)
master_2024 = master_df[master_df['year']==2024].copy()
key_cols = ['year','name','code']
compare_cols = [c for c in repro_2024.columns if c in master_2024.columns and c not in key_cols]
merged = master_2024[key_cols + compare_cols].merge(
    repro_2024[key_cols + compare_cols], on=key_cols, how='inner', suffixes=('_m','_r')
)
print(f"Rows matched: {len(merged)}")
for col in compare_cols:
    m = merged[f"{col}_m"]
    r = merged[f"{col}_r"]
    if m.dtype == object:
        m = m.replace('\\N', pd.NA)
    if r.dtype == object:
        r = r.replace('\\N', pd.NA)
    if col in ['position','grid','laps','rank']:
        m = pd.to_numeric(m, errors='coerce')
        r = pd.to_numeric(r, errors='coerce')
    if col in ['milliseconds','sprint_results_milliseconds']:
        m = pd.to_numeric(m, errors='coerce')
        r = pd.to_numeric(r, errors='coerce')
    if col in ['q1','q2','q3']:
        m = m.apply(normalize_quali)
        r = r.apply(normalize_quali)
    both_na = m.isna() & r.isna()
    both_val = (~m.isna()) & (~r.isna())
    matches = both_na.copy()
    matches[both_val] = (m[both_val] == r[both_val])
    rate = matches.mean()*100
    print(f"{col}: match {rate:.1f}% ({matches.sum()}/{len(matches)})")
    if rate < 50:
        mm = merged.loc[~matches, ['name','code', f"{col}_m", f"{col}_r"]].head(5)
        print(mm)
print("Done.")


Rows matched: 479
resultId: match 0.0% (0/479)
                 name code  resultId_m resultId_r
0  Bahrain Grand Prix  VER       26286       <NA>
1  Bahrain Grand Prix  PER       26287       <NA>
2  Bahrain Grand Prix  SAI       26288       <NA>
3  Bahrain Grand Prix  LEC       26289       <NA>
4  Bahrain Grand Prix  RUS       26290       <NA>
raceId: match 0.0% (0/479)
                 name code  raceId_m raceId_r
0  Bahrain Grand Prix  VER      1121     <NA>
1  Bahrain Grand Prix  PER      1121     <NA>
2  Bahrain Grand Prix  SAI      1121     <NA>
3  Bahrain Grand Prix  LEC      1121     <NA>
4  Bahrain Grand Prix  RUS      1121     <NA>
driverId: match 100.0% (479/479)
constructorId: match 99.6% (477/479)
grid: match 97.7% (468/479)
position: match 90.0% (431/479)
points: match 100.0% (479/479)
laps: match 100.0% (479/479)
time: match 16.1% (77/479)


KeyError: "None of [Index([-2, -1, -1, -1, -1, -1, -1, -1, -1, -1,\n       ...\n       -1, -1, -1, -1, -1, -1, -2, -2, -2, -2],\n      dtype='object', length=479)] are in the [index]"

: 